# ML Assignment 2 - Classification Models

## 1. Load the dataset
Uses the local CSV in `data/` if it exists, otherwise downloads it from UCI and saves a copy.

In [15]:
from pathlib import Path
from urllib.error import URLError

import pandas as pd

LOCAL_PATH = Path("data/breast_cancer_wisconsin.csv")
DATA_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/breast-cancer-wisconsin/wdbc.data"

# raw file has no header - id, diagnosis, then mean/se/worst for each measurement
MEASUREMENTS = ["radius", "texture", "perimeter", "area", "smoothness",
                "compactness", "concavity", "concave_points", "symmetry", "fractal_dimension"]
COLUMNS = ["id", "diagnosis"] + [f"{m}_{stat}" for stat in ("mean", "se", "worst") for m in MEASUREMENTS]


def load_dataset(local_path=LOCAL_PATH, url=DATA_URL):
    local_path = Path(local_path)

    if local_path.exists():
        print(f"Loading local file: {local_path}")
        df = pd.read_csv(local_path)
    else:
        print(f"Local file not found, downloading from: {url}")
        try:
            df = pd.read_csv(url, header=None, names=COLUMNS)
        except (URLError, OSError) as err:
            raise RuntimeError(
                f"Could not find {local_path} and the download failed ({err}). "
                "Place the CSV at that path manually."
            ) from err
        local_path.parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(local_path, index=False)
        print(f"Saved a local copy to: {local_path}")

    # in case the local copy is the Kaggle version (extra Unnamed column, spaces in names)
    df = df.loc[:, ~df.columns.str.startswith("Unnamed")]
    df.columns = df.columns.str.strip().str.replace(" ", "_")
    return df


df = load_dataset()
print(df.shape)
df.head()

Loading local file: data/breast_cancer_wisconsin.csv
(569, 32)


,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave_points_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave_points_worst,symmetry_worst,fractal_dimension_worst
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


## 2. Features and target
- Drop `id` since it's just a patient ID.
- Encode `diagnosis`: M = 1, B = 0.

In [16]:
df = df.drop(columns=["id"])

df["diagnosis"] = df["diagnosis"].map({"M": 1, "B": 0})
assert df["diagnosis"].notna().all(), "diagnosis should only contain M or B"

X = df.drop(columns=["diagnosis"])
y = df["diagnosis"]

print("Features:", X.shape)
print("Target counts (1 = Malignant, 0 = Benign):")
print(y.value_counts())

Features: (569, 30)
Target counts (1 = Malignant, 0 = Benign):
diagnosis
0    357
1    212
Name: count, dtype: int64


## 3. Train/test split
80/20 split, stratified so both parts keep the same class ratio.

The test part is saved without scaling as `test_data.csv`. This is the file to upload in the Streamlit app. Scaling is done later inside the model pipelines so it's only fitted on the training data.

In [17]:
from sklearn.model_selection import train_test_split

RANDOM_STATE = 29
TEST_SIZE = 0.2

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print("Malignant share -> train: %.3f | test: %.3f" % (y_train.mean(), y_test.mean()))

test_data = X_test.copy()
test_data["diagnosis"] = y_test
test_data.to_csv("test_data.csv", index=False)
print("Saved test_data.csv:", test_data.shape)

Train: (455, 30), Test: (114, 30)
Malignant share -> train: 0.374 | test: 0.368
Saved test_data.csv: (114, 31)


## 4. Models
Logistic Regression, KNN and SVM are sensitive to feature scale, so they go in a pipeline with `StandardScaler`. The scaler gets saved along with the model, which means the app can use the raw CSV directly.

Decision Tree and Naive Bayes don't need scaling. `probability=True` on SVC is needed for AUC.

In [18]:
import warnings

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# sklearn 1.9 deprecates SVC(probability=True); still works on the pinned version
warnings.filterwarnings("ignore", message="The `probability` parameter", category=FutureWarning)

models = {
    "Logistic Regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000)),
    "Decision Tree": DecisionTreeClassifier(max_depth=5, random_state=RANDOM_STATE),
    "KNN": make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=7)),
    "Naive Bayes": GaussianNB(),
    "Support Vector Machine": make_pipeline(
        StandardScaler(), SVC(kernel="rbf", probability=True, random_state=RANDOM_STATE)
    ),
}

## 5. Train and evaluate on the test set
Malignant (1) is the positive class for precision, recall and F1. AUC uses the predicted probability of class 1.

In [19]:
from sklearn.metrics import (accuracy_score, roc_auc_score, precision_score,
                             recall_score, f1_score, matthews_corrcoef, confusion_matrix)


def evaluate(model, X, y):
    y_pred = model.predict(X)
    y_score = model.predict_proba(X)[:, 1]  # P(malignant)
    return {
        "Accuracy": accuracy_score(y, y_pred),
        "AUC": roc_auc_score(y, y_score),
        "Precision": precision_score(y, y_pred),
        "Recall": recall_score(y, y_pred),
        "F1": f1_score(y, y_pred),
        "MCC": matthews_corrcoef(y, y_pred),
    }


results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    results[name] = evaluate(model, X_test, y_test)
    tn, fp, fn, tp = confusion_matrix(y_test, model.predict(X_test)).ravel()
    print(f"{name:<24} TN={tn:<3} FP={fp:<3} FN={fn:<3} TP={tp}")

comparison = pd.DataFrame(results).T.round(4)
comparison.index.name = "ML Model Name"
comparison

Logistic Regression      TN=72  FP=0   FN=0   TP=42
Decision Tree            TN=68  FP=4   FN=0   TP=42
KNN                      TN=72  FP=0   FN=0   TP=42
Naive Bayes              TN=69  FP=3   FN=3   TP=39
Support Vector Machine   TN=71  FP=1   FN=1   TP=41


,Accuracy,AUC,Precision,Recall,F1,MCC
ML Model Name,,,,,,
Logistic Regression,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000
Decision Tree,0.9649,0.9722,0.9130,1.0000,0.9545,0.9286
KNN,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000
Naive Bayes,0.9474,0.9873,0.9286,0.9286,0.9286,0.8869
Support Vector Machine,0.9825,0.9997,0.9762,0.9762,0.9762,0.9623


## 6. Cross-validation
The test set is only 114 rows and two models get a perfect score on it, so I'm also running 5-fold stratified CV on the training set to see how stable the scores are.

In [20]:
from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold, cross_validate

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_metrics = {"Accuracy": "accuracy", "AUC": "roc_auc", "F1": "f1", "MCC": "matthews_corrcoef"}

cv_rows = {}
for name, model in models.items():
    scores = cross_validate(clone(model), X_train, y_train, cv=cv, scoring=cv_metrics)
    cv_rows[name] = {m: f"{scores['test_' + m].mean():.3f} ± {scores['test_' + m].std():.3f}"
                     for m in cv_metrics}

cv_summary = pd.DataFrame(cv_rows).T
cv_summary.index.name = "ML Model Name"
cv_summary

,Accuracy,AUC,F1,MCC
ML Model Name,,,,
Logistic Regression,0.965 ± 0.018,0.990 ± 0.009,0.952 ± 0.024,0.925 ± 0.038
Decision Tree,0.925 ± 0.034,0.895 ± 0.032,0.896 ± 0.046,0.841 ± 0.074
KNN,0.965 ± 0.025,0.987 ± 0.018,0.951 ± 0.035,0.925 ± 0.055
Naive Bayes,0.930 ± 0.013,0.983 ± 0.009,0.903 ± 0.018,0.849 ± 0.028
Support Vector Machine,0.969 ± 0.026,0.992 ± 0.011,0.959 ± 0.035,0.934 ± 0.056


## 7. Save the models
Each fitted model (with its scaler where used) is saved to `model/` for the Streamlit app. Also saving the comparison table.

In [21]:
import joblib

MODEL_DIR = Path("model")
MODEL_DIR.mkdir(exist_ok=True)

for name, model in models.items():
    file_name = name.lower().replace(" ", "_") + ".joblib"
    joblib.dump(model, MODEL_DIR / file_name)
    print("Saved", MODEL_DIR / file_name)

comparison.to_csv(MODEL_DIR / "model_comparison.csv")
print("Saved", MODEL_DIR / "model_comparison.csv")

Saved model/logistic_regression.joblib
Saved model/decision_tree.joblib
Saved model/knn.joblib
Saved model/naive_bayes.joblib
Saved model/support_vector_machine.joblib
Saved model/model_comparison.csv
